In [69]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

# 1. Load and Filter
results = pd.read_csv("../package_metadata/results_all.csv")

config = {
    "explainer_total": "shapiq_kernel", 
    "task": "regression",
    "model_name": "nn",
}
target_kernels = ["kt_matern", "kt_inverse_multiquadric", "kt_gaussian"]

results_filtered = results[
    (results["explainer_total"] == config["explainer_total"]) &
    (results["task"] == config["task"]) &
    (results["model_name"] == config["model_name"]) &
    (results["method_total"].isin(target_kernels))
].copy()

def perform_rigorous_analysis(df, metric, lower_is_better=True):
    print(f"\n{'='*40}\nSTATISTICAL ANALYSIS FOR: {metric.upper()}\n{'='*40}")
    
    # Step A: Check if metric ends with _std
    if metric.endswith('_std'):
        # For std metrics, calculate std across repetitions
        # Remove _std suffix to get the base metric name
        base_metric = metric.replace('_std', '')
        # Check if it's top_k__std (with double underscore)
        if base_metric.endswith('_'):
            base_metric = base_metric[:-1]
        
        # Calculate std across repetitions for each block
        df_agg = df.groupby(["dataset_name", "compression_coefficient", "method_total"])[base_metric].std().reset_index()
        df_agg = df_agg.rename(columns={base_metric: metric})
    else:
        # For regular metrics, aggregate by mean
        df_agg = df.groupby(["dataset_name", "compression_coefficient", "method_total"])[metric].mean().reset_index()
    
    # Step B: Pivot for the tests
    pivot_df = df_agg.pivot_table(index=["dataset_name", "compression_coefficient"], 
                                  columns="method_total", values=metric).dropna()
    
    # print(pivot_df)
    
    # Step C: Friedman Test
    stat, p_friedman = friedmanchisquare(*[pivot_df[col].values for col in target_kernels])
    print(f"Friedman Test p-value: {p_friedman:.4f}")
    
    # Step D: Calculate Average Ranks
    ranks = pivot_df.rank(axis=1, ascending=lower_is_better)
    avg_ranks = ranks.mean().sort_values()
    print("\nAverage Ranks (1.0 is the best possible):")
    print(avg_ranks)
    
    # Step E: Post-hoc Wilcoxon Tests 
    if p_friedman < 0.05:
        print("\nPairwise Wilcoxon Tests (with Bonferroni Correction):")
        pairs = list(combinations(target_kernels, 2))
        alpha_adj = 0.05 / len(pairs) # Bonferroni correction for 3 comparisons
        
        comparison_table = []
        for (m1, m2) in pairs:
            _, p_val = wilcoxon(pivot_df[m1], pivot_df[m2])
            is_sig = p_val < alpha_adj
            
            mean_diff = pivot_df[m1].mean() - pivot_df[m2].mean()
            if is_sig:
                if lower_is_better:
                    winner = m1 if mean_diff < 0 else m2
                else:
                    winner = m1 if mean_diff > 0 else m2
            else:
                winner = "No significant difference"
                
            comparison_table.append({
                "Comparison": f"{m1} vs {m2}",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if is_sig else "No",
                "Winner": winner
            })
        print(pd.DataFrame(comparison_table))
    else:
        print("\nConclusion: No statistically significant difference between kernels.")

# Execute for both metrics
# perform_rigorous_analysis(results_filtered, "mae", lower_is_better=True)
# perform_rigorous_analysis(results_filtered, "top_k", lower_is_better=False)
perform_rigorous_analysis(results_filtered, "mae_std", lower_is_better=True)
perform_rigorous_analysis(results_filtered, "top_k__std", lower_is_better=True)


STATISTICAL ANALYSIS FOR: MAE_STD
Friedman Test p-value: 0.0670

Average Ranks (1.0 is the best possible):
method_total
kt_matern                  1.835294
kt_inverse_multiquadric    2.000000
kt_gaussian                2.164706
dtype: float64

Conclusion: No statistically significant difference between kernels.

STATISTICAL ANALYSIS FOR: TOP_K__STD
Friedman Test p-value: 0.0013

Average Ranks (1.0 is the best possible):
method_total
kt_matern                  1.741176
kt_inverse_multiquadric    2.000000
kt_gaussian                2.258824
dtype: float64

Pairwise Wilcoxon Tests (with Bonferroni Correction):
                               Comparison p-value Significant  \
0    kt_matern vs kt_inverse_multiquadric  0.1738          No   
1                kt_matern vs kt_gaussian  0.0046         Yes   
2  kt_inverse_multiquadric vs kt_gaussian  0.2186          No   

                      Winner  
0  No significant difference  
1                  kt_matern  
2  No significant difference  

/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_18684/3232662099.py:7: DtypeWarning: Columns (30,31,32,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  results = pd.read_csv("../package_metadata/results_all.csv")
